In [36]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path


folder = Path(r"C:\Users\Asus\Documents\Programming\Programming\Data analysis-technical complex\.ipynb_checkpoints\Kaggle dataset")

#همسانسازی داده ها
gold = pd.read_csv(folder / "gold18.csv")[['Date', 'Open']]
oil = pd.read_csv(folder / "crude_oil15to24.csv")[['Date', 'Price']]
dollar = pd.read_csv(folder / "USDIRR(1390-1402).csv")[['Date', 'Opening']]
gold = gold.rename(columns={'Open': 'Gold_Price'})
oil = oil.rename(columns={'Price': 'Oil_Price($)'})
dollar = dollar.rename(columns={'Opening': 'Dollar_Price'})

#تبدیل داده ها
datasets = [gold, oil, dollar]
price_cols = ['Gold_Price', 'Oil_Price($)', 'Dollar_Price']
for data in datasets:
    data['Date'] = pd.to_datetime(data['Date'], errors='coerce')
for data, col in zip(datasets, price_cols):
    data[col] = pd.to_numeric(data[col].astype(str).str.replace(',', '', regex=False),errors='coerce')
    
#محاسبه میانگین سالانه
merged = gold.merge(oil, on='Date', how='outer')
merged = merged.merge(dollar, on='Date', how='outer')
merged = merged.sort_values('Date')
merged['year'] = merged['Date'].dt.year

df = merged.groupby('year')[price_cols].mean().reset_index()

# محاسبه ریالی نفت
df['Oil_Price(IRR)'] = df['Oil_Price($)'] * df['Dollar_Price']

# محاسبه درصد تغییر سالانه
change_cols = ['Gold_change%','Oil_change_$%','Dollar_Change %','Oil_change%']
source_cols = ['Gold_Price', 'Oil_Price($)','Dollar_Price','Oil_Price(IRR)']
for col, price in zip(change_cols, source_cols):
    df[col] = df[price].pct_change() * 100

# ذخیره سازی 
csv_path = folder / "Gold_Oil_$ analysis.csv"
excel_path = folder / "Gold_Oil_$ analysis.xlsx"
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
describe = df[['Gold_Price', 'Oil_Price($)', 'Dollar_Price', 'Oil_Price(IRR)']].describe().round(2)

with pd.ExcelWriter(excel_path) as writer:
    df.to_excel(writer, sheet_name='Data', index=False)
    describe.to_excel(writer, sheet_name='Describe')
print(df.round(2).to_string(index=False))
print(f"\nFiles saved in: {folder}")

#روند تغییر سالانه قیمت ها
price_cols = ['Gold_Price', 'Oil_Price(IRR)', 'Dollar_Price']
df.plot( x='year', y=price_cols, marker='o', figsize=(12, 6), logy=True)
plt.title('Average Yearly Prices')
plt.xlabel('Year')
plt.ylabel('Average Price (log scale)')
plt.legend(['Gold', 'Oil (IRR)', 'Dollar'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# درصد نوسان سالانه
df.plot(x='year',y=['Gold_change%', 'Oil_change_$%', 'Dollar_Change %'],kind='bar', figsize=(12, 6), width=0.8)
plt.title('Annual Price Changes')
plt.xlabel('Year')
plt.ylabel('Change (%)')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# تاثیر نوسان قیمت دلار بر قیمت نفت در ایران
effect_cols = ['Oil_change_$%', 'Dollar_Change %', 'Oil_change%']
effect = df[['year'] + effect_cols].dropna()
effect.plot(x='year', y=effect_cols,kind='bar', figsize=(12, 6), width=0.8)
plt.axhline(y=0, color='black', linewidth=0.8)
plt.title('Exchange Rate Effect on Oil Price in Iran')
plt.xlabel('Year')
plt.ylabel('Annual Change (%)')
plt.legend(['Oil_Change (USD)','Dollar_Change',' Oil_Change (IRR)'])
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# بررسی روند بلند مدت بر اساس شاخص 
index_cols = ['Gold_Price', 'Oil_Price(IRR)','Dollar_Price']
common_years = df.dropna(subset=index_cols)
base_year = int(common_years.iloc[0]['year'])
base_prices = common_years.iloc[0][index_cols]
for col in index_cols:
    df[col + '_Index'] = df[col] / base_prices[col] * 100
df.plot( x='year', y=[col + '_Index' for col in index_cols],marker='o', figsize=(12, 6))
plt.title(f'Annual Price Trend Index ({base_year} = 100)')
plt.xlabel('Year')
plt.ylabel('Price Index')
plt.legend(['Gold', 'Oil (IRR)', 'Dollar'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# بررسی میانگین متحرک 3 ساله
cols = ['Gold_Price', 'Oil_Price(IRR)','Dollar_Price']
for col in cols:
    df[col + '_MA'] = df[col + '_Index'].rolling(3).mean()
df.plot( x='year',y=['Gold_Price_MA', 'Oil_Price(IRR)_MA', 'Dollar_Price_MA'],figsize=(12, 6),marker='o')

plt.title('3 Year Moving Average')
plt.xlabel('Year')
plt.ylabel('Price Index (Base Year = 100)')
plt.legend(['Gold', 'Oil (IRR)', 'Dollar'])
plt.tight_layout()
plt.show()

#تحلیل همبستگی داده ها 
source_cols = ['Gold_Price', 'Oil_Price($)','Dollar_Price','Oil_Price(IRR)']
corr = df[source_cols].corr()
print(corr.round(2))
sns.heatmap(corr,annot=True,cmap='coolwarm',vmin=-1,vmax=1,center=0)   
plt.title('Correlation Between Gold, Oil and Dollar')
plt.tight_layout()
plt.show()

# بررسی نوسان درصد تغییرات سالاته
volatility = df[['Gold_change%', 'Oil_change_$%', 'Dollar_Change %']].std()
volatility.index = ['Gold', 'Oil (USD)', 'Dollar']

volatility.plot(kind='bar',figsize=(8, 5),rot=0)
plt.title('Standard Deviation')
plt.xlabel('Gold_oil_dollar')
plt.ylabel(' Std of Annual Change (%)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

#بررسی تاثیر نرخ ارز بر رشد طلا 
df.plot(x='Dollar_Price',y='Gold_Price',kind='scatter',figsize=(8, 5))
plt.show()

#بررسی میانگین متحرک 
df['Gold'] = (df['Gold_Price'].rolling(3).mean())
df.plot(x='year',y=['Gold_Price', 'Gold'],marker='o')

plt.title('Gold Price/3 Years Moving Average')
plt.xlabel('Year')
plt.ylabel('Gold Price')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(df[['year', 'Gold_Price', 'Gold']].tail(8).round(2))

#بررسی روند احتمالی اینده
forecast_cols = {'Gold_Price': 'Gold Price','Oil_Price($)': 'Oil Price (USD)','Dollar_Price': 'Dollar Exchange Rate'}

history = df.sort_values('year').tail(10).copy()
for col, title in forecast_cols.items():
    data = history[['year', col]].dropna()
    if len(data) < 3 or (data[col] <= 0).any():
        print(f'no valid data for {title} ')
        continue

    years = data['year'].to_numpy()
    prices = data[col].to_numpy()

    coefficients = np.polyfit(years, np.log(prices), 1)

    future_years = np.arange(int(years.max()) + 1,int(years.max()) + 11 )

    predicted = np.exp(np.polyval(coefficients, future_years))

    plt.figure(figsize=(11, 6))
    plt.plot(years, prices,marker='o',label='Historical data')

    plt.plot(future_years, predicted,linestyle='--',marker='o',label='Forecast')
    plt.axvline(years.max(),linestyle=':', color='gray',label='Forecast starts')
    plt.title(f'{title}: 10-Year History and Forecast')
    plt.xlabel('Year')
    plt.ylabel('Price')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

'''تحلیل نوسانات نرخ ارز بر قیمت طلا و نفت در ایران (۲۰۱۵–۲۰۲۴)

بررسی داده‌های سال‌های ۲۰۱۵ تا ۲۰۲۴ نشان می‌دهد که قیمت طلا و نفت در ایران، علاوه بر تحولات بازارهای جهانی، به‌شدت از نوسانات نرخ ارز تأثیر گرفته است. در این دوره، عواملی مانند تحریم‌ها، شیوع کرونا، جنگ روسیه و اوکراین و تغییرات عرضه و تقاضای جهانی، نقش مهمی در نوسانات قیمت‌ها داشته‌اند.
در سال ۲۰۱۸، هم‌زمان با بازگشت تحریم‌های آمریکا، نرخ دلار حدود ۱۱۰٪ و قیمت طلا در ایران حدود ۱۰۷٪ افزایش یافت. در سال ۲۰۲۰ نیز با وجود افت حدود ۳۳٪ قیمت جهانی نفت، قیمت محاسبه‌شده نفت به پول ایران ۱۲٪ رشد کرد؛ زیرا افزایش نرخ دلار بخشی از کاهش قیمت جهانی را جبران کرد. این روند در سال ۲۰۲۴ هم دیده می‌شود: با وجود کاهش حدود ۲٫۵٪ قیمت دلاری نفت، افزایش ۱۲٫۵٪ نرخ دلار باعث شد قیمت نفت به پول ایران نزدیک به ۱۰٪ افزایش یابد.
بر اساس داده‌های سال‌های ۲۰۱۵ تا ۲۰۲۴، قیمت طلا در ایران حدود ۳٬۱۰۱٪، نرخ دلار حدود ۱٬۵۴۹٪، قیمت جهانی نفت حدود ۵۵٫۲٪ و قیمت نفت به پول ایران حدود ۲٬۴۵۹٪ افزایش یافته است.میانگین متحرک سه‌ساله اثر جهش‌های ناگهانی قیمت را در طول چند سال پخش می‌کند؛ بنابراین، افزایش شدید قیمت طلا در سال‌های ۲۰۲۰ و ۲۰۲۳ و نفت به پول ایران در سال‌های ۲۰۱۸، ۲۰۲۱ و ۲۰۲۲، با بالا بردن میانگین سال جاری و سال‌های بعد، روند صعودی آن را تقویت می‌کند.د. 
بر اساس مدل داده‌های ده سال اخیر، برآورد می‌شود طی سه سال آینده تا ۲۰۲۷، قیمت طلا حدود ۲۹۵٪، دلار حدود ۲۴۱٪ و نفت جهانی حدود ۳۰٪ افزایش یابد؛ در افق ده‌ساله تا ۲۰۳۴، افزایش برآوردشده به‌ترتیب حدود ۸٬۰۰۰٪، ۳٬۹۰۰٪ و ۱۰۹٪ است. این نکته را باید در نظر داشت که مسیر قیمت‌ها به عواملی مانند تحریم‌ها و روابط سیاسی ایران، تنش‌های منطقه‌ای، سیاست‌های ارزی و پولی، تصمیمات اوپک‌پلاس، رشد اقتصاد جهانی و تغییرات عرضه و تقاضای نفت و طلا وابسته است؛ بنابراین تغییر این شرایط می‌تواند باعث شود قیمت‌های واقعی به‌طور قابل‌توجهی با برآورد مدل تفاوت داشته باشند.
 '''

    


SyntaxError: invalid character '۲' (U+06F2) (2738961411.py, line 190)